In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt  
from sklearn.model_selection import train_test_split

### Implementation (per-leaf mean updates -- standard GBRT behaviour)

In [2]:
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor

In [11]:
class MyGradientBoostingRegressor: 
    """ 
    Gradient Boosting Regressor(least-square loss)
    - using DecisionTreeRegressor as base learner.
    """
    def __init__(self , n_estimators = 100 , learning_rate = 0.1 , max_depth = 3, 
                min_samples_leaf = 1 , subsample = 1.0 , random_state = 42 , verbose = False) -> None:
        self.n_estimators = n_estimators 
        self.learning_rate = learning_rate 
        self.max_depth = max_depth 
        self.min_samples_leaf = min_samples_leaf 
        self.subsample = subsample 
        self.random_state = random_state
        self.verbose = verbose 
        # containers to store learned ensemble 
        self.trees = [] # store the fitted trees
        self.leaf_values = [] # list of dicts:{leaf_id: v_j}
        self.train_score_ = [] # MSE after each iteration
        self.F0 = None # initial constant predictor(mean of target for MSE loss method)

    def fit(self , X , y): 
        # convert X and y to numpy array 
        X = np.asarray(X)
        y = np.asarray(y)

        n_samples = X.shape[0]
        rng = np.random.RandomState(self.random_state)
        
        # 1. initialize the F0(constant model) 
        self.F0 = y.mean()
        # current F0 prediction for all rows 
        F = np.full(shape = n_samples , fill_value = self.F0 , dtype = float)
        
        # 2. iterate from 1 to M(number of base learners)
        for m in range(self.n_estimators): 
            # 2.a: compute the residual [residual = acutal_target - predict]
            residuals = y - F 

            # Optional stochastic boosting: subsample indices (without replacement)
            # if we dont want to use the all rows of X. Use subsample percentage
            if 0 < self.subsample < 1.0: 
                n_sub = max(1 , int(self.subsample * n_samples))
                sample_idx = rng.choice(n_samples , n_sub , replace = False)
            else: # otherwise the take the all rows 
                sample_idx = np.arange(n_samples)

            # 2.b: fit a Decision Tree Regressor with X and residual 
            tree = DecisionTreeRegressor(
                max_depth = self.max_depth , min_samples_leaf = self.min_samples_leaf , random_state = rng.randint(0 , 2**31 - 1) 
            )
            tree.fit(X[sample_idx] , residuals[sample_idx])
            # 2.c: compute per-leaf optimal values v_j = mean residual of samples in that leaf 
            # get all the id's of leaf for the tree we trained 
            leaf_ids = tree.apply(X)
            unique_leaf_ids = np.unique(leaf_ids)

            leaf_value_map = {}
            for leaf_id in unique_leaf_ids: 
                # get all the rows which fall into this leaf 
                mask = (leaf_ids == leaf_id)
                #  mean residual on the training samples that fall into this leaf 
                v_j = residuals[mask].mean()
                leaf_value_map[leaf_id] = float(v_j)

            # save the tree and its leaf-value mapping 
            self.trees.append(tree)   
            self.leaf_values.append(leaf_value_map)

            # update F for all the training samples
            updates = np.fromiter((leaf_value_map[l] for l in leaf_ids) , dtype = float)
            F += self.learning_rate * updates

            # track the loss after update
            mse = mean_squared_error(y , F) 
            self.train_score_.append(mse) 

            if self.verbose: 
                print(f"[iter {m + 1:3d}] train MSE: {mse : .6f}")
        return self

    def predict(self , X): 
        X = np.asarray(X) 
        n = X.shape[0]
        # start from initial constant prediction
        F = np.full(n, self.F0, dtype = float) 

        for tree , leaf_map in zip(self.trees , self.leaf_values): 
            leaf_ids = tree.apply(X) 
            # for any leaf id not seen (shouldn't happen on training but may on new data),
            # default to 0.0 correction
            updates = np.fromiter((leaf_map.get(l, 0.0) for l in leaf_ids), dtype=float)
            F += self.learning_rate * updates

        return F 

    def staged_predict(self , X): 
        X = np.asarray(X)
        n = X.shape[0]
        F = np.full(n , self.F0 , dtype = float)

        yield F.copy()
        for tree , leaf_map in zip(self.trees , self.leaf_values): 
            leaf_ids = tree.apply(X)
            updates = np.fromiter((leaf_map.get(l, 0.0) for l in leaf_ids), dtype = float)
            F += self.learning_rate * updates
            yield F.copy()

In [5]:
from sklearn.datasets import make_regression
X, y = make_regression(n_samples = 500 , n_features = 5, noise = 20.0, random_state = 42)

In [12]:
gbr = MyGradientBoostingRegressor(
    n_estimators = 100,
    learning_rate = 0.1,
    max_depth = 3,
    min_samples_leaf = 5,
    subsample = 0.8,
    random_state = 42,
    verbose = True
)

In [13]:
gbr.fit(X , y)

[iter   1] train MSE:  11798.709082
[iter   2] train MSE:  10364.081118
[iter   3] train MSE:  9083.650822
[iter   4] train MSE:  8034.418065
[iter   5] train MSE:  7133.603896
[iter   6] train MSE:  6399.438603
[iter   7] train MSE:  5766.345460
[iter   8] train MSE:  5215.257472
[iter   9] train MSE:  4756.690627
[iter  10] train MSE:  4348.289335
[iter  11] train MSE:  3990.689378
[iter  12] train MSE:  3649.639189
[iter  13] train MSE:  3345.275055
[iter  14] train MSE:  3090.862084
[iter  15] train MSE:  2856.760727
[iter  16] train MSE:  2630.396717
[iter  17] train MSE:  2431.906613
[iter  18] train MSE:  2252.374377
[iter  19] train MSE:  2091.665999
[iter  20] train MSE:  1950.561967
[iter  21] train MSE:  1817.905480
[iter  22] train MSE:  1702.998714
[iter  23] train MSE:  1595.305113
[iter  24] train MSE:  1501.714822
[iter  25] train MSE:  1416.278568
[iter  26] train MSE:  1330.261740
[iter  27] train MSE:  1251.602212
[iter  28] train MSE:  1185.217990
[iter  29] train M

In [14]:
y_pred = gbr.predict(X)

In [15]:
mean_squared_error(y , y_pred)

246.95368447784557